# Noor's service night: testing USR 2.0 on a real YouTube clip

This notebook runs the authors' model, **USR 2.0** ([paper](https://arxiv.org/abs/2602.19316) · [code](https://github.com/ahaliassos/usr2)), on a short clip from
[this YouTube video](https://www.youtube.com/watch?v=0kconvB5dlI). We'll read it three ways:

* **a**: audio only (the regular customer who speaks up)
* **v**: lips only (Noor while the train goes over)
* **av**: both (Noor on a quiet day)

Then we play the train: we add crowd noise and check whether the lips rescue the audio.

### How to use Colab (2 minutes)
* A notebook is a stack of **cells**. Grey boxes are **code cells**; this one is a **text cell**.
* To run a code cell, click it and press **Shift + Enter** (or click ▶ on its left). Go **top to bottom, one at a time**.
* While a cell runs you see a spinning circle. When it finishes, a number appears in the brackets: `[3]`.
* A line starting with `!` is a **command** sent to the terminal (Sami's *pass*). Everything else is Python.
* The machine is rented and wiped when you close the tab or stay idle (~90 min on the free tier). If that happens, run the cells again from Step 1.
* If a cell ends in red text, that's a **traceback**. Read the **last line** first, then see *Troubleshooting* at the bottom.

### Time plan (first time through)
| Step | What | Time |
|---|---|---|
| 0 | Read the principles (Part A) | 10–15 min |
| 1 | Rent the kitchen: switch on a GPU | 2–5 min |
| 2 | Buy the cookbook: `git clone` | < 1 min |
| 3 | Stock the shelf: install packages (+ one restart) | 4–8 min |
| 4 | Fetch the chef's brain: checkpoint download | 2–6 min (Huge) |
| 5 | Pick and cut the clip; type what was said | 5–15 min |
| 6 | Service: run **av**, look at the mouth crop | 2–5 min |
| 7 | Run **a** and **v**, score them (WER) | 3–8 min |
| 8 | The train: add babble noise, re-test **a** vs **av** | 5–10 min |
| 9 | Clean up | 1 min |
| | **Total** | **~35–75 min** |

These are estimates for a free Colab T4 GPU. The notebook prints real timings as you go. Without a GPU, each model run takes several minutes instead of about one.

---
# Part A: the principles, in Noor's terms (read before you click)

### A1. The chef (what the model is)
USR 2.0 is **one unified model** that does all three jobs: ASR (audio), VSR (lips) and AVSR (both). Inside: **two doors, one hall, two exits.**

* **Doors (frontends):** two small ResNet-18 networks. One turns each 88×88 grey mouth tile into numbers; the other turns raw 16 kHz audio into numbers, cut to **25 slices a second** so each slice sits beside one video frame (*two rowers, one stroke*). With both senses, the two lists are joined side by side and projected: **fusion**.
* **Hall (encoder):** a Transformer. Every frame looks at every other frame (**attention**). That's how the *twins* (p/b/m, the **visemes**) are told apart: by **context**.
* **Exits:** the **CTC head** (*the line judge*: calls every frame at once; fast and sturdy, a bit deaf to grammar) and the **attention decoder** (*the commentator*: one word at a time; fluent, but it can drift).
* **At service** (`demo.py`), a **beam search** keeps **40** candidate sentences and blends the exits: **90 % commentator, 10 % line judge** (`ctc_weight=0.1`). Output is word pieces from a 1,000-token vocabulary, glued into CAPITALS with no punctuation.

### A2. What the paper changed: the coach's notes (training only)
USR 2.0 learns from a few labelled clips plus a huge pile of unlabelled ones. A **teacher** (a slow average, or EMA, of the student's own weights) watches the clean audio and video and writes **pseudo-labels**. The **student** gets the masked, single-sense view and must match them. Only confident notes (> 0.8) count.

* **Old way (USR 1):** the commentator's note was written **autoregressively**, one word at a time, at *every* training step. That was slow. Worse, when it drifted on a long or noisy clip, the student learned the drift, and since the teacher *is* the student's average, the drift fed itself.
* **New trick: CTC-driven teacher forcing.** Take the line judge's quick **greedy** call, collapse repeats and blanks, and feed it to the commentator *as if it were the true sentence*. The commentator's note comes out in **one pass**. The two notes now line up word for word, so the student's decoder learns from both, and the line judge's sturdiness rubs off on the commentator.
* **Mixed sampling:** at each step a coin flip (p = 0.5) chooses between the new mode and the old slow mode. That keeps the commentator practised at working alone (**exposure bias**: *a player who never played a real match*).
* **Result:** about **2× faster training** (faster steps, and 50 epochs instead of 75). The decoder is also **much steadier on long clips, noise and unfamiliar video**.

**Why this matters to us even though we only do inference:** we never train. But the brain we load *was* trained this way, so we inherit its steadiness. The paper shows USR 2.0 stays good with small beams, even greedy (1 candidate). That's the speed knob from Chapter 5.

### A3. What the chef has eaten (training data) → how to pick a clip
* Labelled: **LRS3** (TED talks) and **LRS2** (BBC TV). Unlabelled: English-only **VoxCeleb2** (interviews) and **AVSpeech** (YouTube). All **English**.
* The longest *labelled* clip was **155 frames (~6 s)**; unlabelled clips went up to 600 frames (24 s).
* Preprocessing expects one face, 25 fps, 16 kHz audio, and a mouth crop that MediaPipe can find.

➡️ **Pick 5–15 s of English, one person, face roughly frontal, mouth visible, no camera cuts.** Anything else is **out of distribution**, *a French chef asked for sushi*.

### A4. What's at stake: the numbers we're testing against
| Situation (from the paper) | WER |
|---|---|
| Lips only, Huge, LRS3 test (in-distribution) | **17.6 %** |
| Lips only, Base, WildVSR (wild internet video, greedy) | **73.7 %** |
| Audio only / AV, Huge, LRS3 test | **0.9 % / 0.8 %** |
| Audio only, Base, babble noise at **0 dB** | **44.0 %** |
| Audio + lips, Base, babble noise at **0 dB** | **14.0 %** |
| AV, Base, AVSpeech (YouTube-like, greedy) | **25.0 %** (USR 1: 34.7 %) |

So our hypotheses, in Noor's words:
1. **Clean audio:** `a` and `av` should be nearly perfect on a clear English speaker.
2. **Lips only:** `v` will be much worse. A YouTube clip sits somewhere between TED talks (~18 %) and the wild (~74 %).
3. **The train passes (0 dB babble):** `a` should collapse and `av` should hold. **This is the whole reason the cafe needs the app.**

⚠️ One 10-second clip is a *taste*, not a *blood test*. With 25 words, each mistake costs 4 points of WER. A real **baseline** needs dozens of clips (Chapter 8).

---
# Step 1: Rent the kitchen (switch on a GPU) · 2–5 min

**In the story:** the CPU is *one star striker*; the GPU is *a hundred decent players* doing thousands of small multiplications at once. The chef runs on either, but the Huge brain is ~950 million dials, so on the striker you wait minutes instead of seconds.

**Do this first:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.** Then run the cell below.

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode == 0:
    print("GPU ready:", r.stdout.strip())
else:
    print("No GPU. Go to Runtime > Change runtime type > T4 GPU > Save, then run this cell again.")
    print("(Free GPUs aren't always available. If none is offered, you can continue on CPU; it's just slower.)")

# Step 2: Buy the cookbook (`git clone`) · < 1 min

**In the story:** `git clone` copies the authors' code onto our rented machine. We never rewrite their recipes; we only *run* them.

In [ ]:
%cd /content
![ -d usr2 ] || git clone --depth 1 https://github.com/ahaliassos/usr2.git
%cd /content/usr2
!ls

# Step 3: Stock the shelf (install packages) · 4–8 min

**In the story:** `pip` is *the grocer*, and `requirements.txt` is *the shopping list*. Colab already has PyTorch, the heavy ingredient.

**One spoiled ingredient:** `demo.py` opens videos with `torchvision.io.read_video`. The torchvision team deprecated that function in version 0.22 and **removed it after 0.25**. On a new Colab, the shelf may have a version without it, and `demo.py` would crash with `AttributeError: ... has no attribute 'read_video'`. The cell below checks. If the function is missing, it installs the last matching trio that still has it (torch 2.10 / torchvision 0.25 / torchaudio 2.10).

If it prints **RESTART NEEDED**: click **Runtime → Restart session**, wait a few seconds, then go straight to **Step 3b**. Don't run Steps 1–3 again.

In [ ]:
import subprocess, sys
probe = subprocess.run([sys.executable, "-c",
                        "import torchvision, torchvision.io as io; print(torchvision.__version__, hasattr(io, 'read_video'))"],
                       capture_output=True, text=True).stdout.split()
print("torchvision on this machine:", probe)
NEED_PIN = not (len(probe) == 2 and probe[1] == "True")
PINS = "torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0" if NEED_PIN else ""
print("Installing the shopping list", "+ older torch trio" if NEED_PIN else "", "...")
!pip install -q {PINS} -r /content/usr2/requirements.txt jiwer gdown "yt-dlp[default]"
if NEED_PIN:
    print("\n>>> RESTART NEEDED: Runtime > Restart session, then run Step 3b. <<<")
else:
    print("\nDone. No restart needed. Run Step 3b.")

### Step 3b: Check the shelf (run after the restart, if there was one)
Expected: `read_video: True`, `cuda: True` (if you have a GPU) and a mediapipe version. Pink *pip dependency conflict* warnings during the install are normal on Colab and can be ignored.

In [ ]:
import torch, torchvision, torchaudio, mediapipe, torchvision.io as tio
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| torchaudio", torchaudio.__version__)
print("mediapipe", mediapipe.__version__)
print("read_video:", hasattr(tio, "read_video"), "| cuda:", torch.cuda.is_available())

# Step 4: Fetch the chef's saved brain (checkpoint) · 2–6 min

**In the story:** the trained chef comes as a file, a **checkpoint**, with every dial saved. We load it into a skeleton of code, the **backbone**. *A chef's saved brain only fits the body it was trained in.* That's why `MODEL` below sets both the file and the matching `model/backbone=...` setting (a **Hydra override**, *"no onions"*).

| Choice | Parameters | Download (my estimate, float32) | Lips-only WER on LRS3 |
|---|---|---|---|
| `"huge"` (recommended by the authors) | 953 M | ~3.8 GB | 17.6 % |
| `"baseplus"` (light fallback) | 171 M | ~0.7 GB | 24.8 % |

Small correction to Chapter 7 of the story: the Huge brain is a few *gigabytes*, not a few hundred megabytes. It still fits a free T4 (15 GB of GPU memory, ~12 GB of RAM).

If the download fails with *"Too many users have viewed or downloaded this file"*: open the Google Drive link printed by the cell, click **Download**, then drag the `.pth` file into Colab's **Files** panel (folder icon on the left) under `/content/ckpt/`, named as printed. Or switch to `"baseplus"`.

In [ ]:
import os, gdown
MODEL = "huge"          # or "baseplus"
CKPTS = {
    "huge":     ("1LzFOTYu45zCLOHGVLQt7pMGjw6jmmo9Y", "resnet_transformer_huge"),
    "baseplus": ("18vmJjdem5XPOA8bmizybIW5sLJuHMdRR", "resnet_transformer_baseplus"),
}
FILE_ID, BACKBONE = CKPTS[MODEL]
CKPT = f"/content/ckpt/usr2_{MODEL}_high.pth"
os.makedirs("/content/ckpt", exist_ok=True)
if not os.path.exists(CKPT):
    print("Drive link (for a manual download):", f"https://drive.google.com/file/d/{FILE_ID}/view")
    gdown.download(id=FILE_ID, output=CKPT, quiet=False)
print(f"{CKPT}: {os.path.getsize(CKPT)/1e9:.2f} GB  | backbone = {BACKBONE}")

# Step 5: Choose and cut the clip · 5–15 min

**In the story (Chapter 4):** the chef never sees a face, only numbers. It wants **25 frames a second** (the *flipbook*) and **16,000 audio samples a second**, in **mono** (*one microphone*).

**Why cut a short clip:** `demo.py` loads every frame of the video into memory at once. A full 720p video of several minutes would fill the machine's RAM and crash the session. The model was also trained on short sentences (labelled clips up to ~6 s).

### 5a. Pick your 5–15 seconds
Open [the video](https://www.youtube.com/watch?v=0kconvB5dlI) in another tab. Find a stretch where **one person speaks English facing the camera, mouth visible, no cuts or overlays on the mouth**. Write the start and end times below.

*Alternative that teaches the most:* film **yourself** on your phone saying a sentence you wrote down, like Noor in Chapter 7. You'll know the exact words, and it's your own face (no personal-data question). Use the upload route in 5b-Plan B.

In [ ]:
URL   = "https://www.youtube.com/watch?v=0kconvB5dlI"
START = "0:30"   # <-- change: start of your segment (m:ss)
END   = "0:40"   # <-- change: end of your segment (m:ss)

def to_sec(t):
    s = 0.0
    for part in t.split(":"):
        s = s * 60 + float(part)
    return s
print(f"Segment: {START} -> {END} = {to_sec(END) - to_sec(START):.1f} s (aim for 5-15 s)")

### 5b. Download just that segment (Plan A)
YouTube often blocks downloads from cloud machines like Colab. If you see *"Sign in to confirm you're not a bot"* or another error, skip to **Plan B**. Download only for this personal test, and respect YouTube's terms and the speaker.

In [ ]:
!mkdir -p /content/work && rm -f /content/work/raw.*
!yt-dlp --js-runtimes node -f "bv*[height<=720]+ba/b[height<=720]/b" --download-sections "*{START}-{END}" --force-keyframes-at-cuts -o "/content/work/raw.%(ext)s" "{URL}"
!ls -la /content/work

### 5b, Plan B: upload a file instead
Download the video (or film yourself) on your own computer, then run this cell and pick the file. You can upload the full video: Step 5c cuts out your START–END segment. A short file uploads much faster, though. If you upload your **own** recording, first set START and END in 5a to the times *within your file* (for example `0:00` to `0:08`).

In [ ]:
import os, shutil
from google.colab import files
os.makedirs("/content/work", exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
ext = os.path.splitext(name)[1] or ".mp4"
for f in os.listdir("/content/work"):
    if f.startswith("raw."):
        os.remove(os.path.join("/content/work", f))
shutil.move(name, f"/content/work/raw{ext}")
print("Saved as", f"/content/work/raw{ext}")

### 5c. Make it the way the chef likes it: 25 fps, 16 kHz mono, ≤ 720p
This cell also shows you the clip. **Watch it**: is the mouth clearly visible the whole time?

In [ ]:
import glob, subprocess, base64
from IPython.display import HTML, display

def duration(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                          "-of", "csv=p=0", path], capture_output=True, text=True).stdout
    return float(out.strip() or 0)

def show_video(path, width=480):
    data = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls src="data:video/mp4;base64,{data}"></video>'))

RAW  = sorted(glob.glob("/content/work/raw.*"))[0]
CLIP = "/content/work/clip.mp4"
seg = to_sec(END) - to_sec(START)
already_cut = duration(RAW) <= seg + 3          # yt-dlp already cut it
trim = [] if already_cut else ["-ss", str(to_sec(START)), "-to", str(to_sec(END))]
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *trim, "-i", RAW,
                "-vf", "scale=-2:'min(720,ih)',fps=25", "-ar", "16000", "-ac", "1",
                "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "aac", CLIP], check=True)
print(f"clip.mp4 ready: {duration(CLIP):.1f} s")
show_video(CLIP)

### 5d. The recipe card: type exactly what is said
This is our **label**, the right answer the taster compares against. Listen to the clip (YouTube's *… → Show transcript* can help, but check it by ear) and type the words. Case and punctuation don't matter; the scoring cell normalises them.

In [ ]:
REFERENCE = "type the exact words spoken in your clip here"

# Step 6: Service night, first order: `av` · 2–5 min

**What happens inside `demo.py`** (the run order matches the story):
1. Read the video; resample to 25 fps and 16 kHz mono; cut the audio to exactly 640 samples per frame (**alignment**).
2. **MediaPipe** finds 478 face points and keeps 68 **landmarks**. The face is aligned to a mean face, smoothed over 12 frames, and a 96×96 square around the lips is cut, then centre-cropped to **88×88**, made grey and **normalised**. That tile is saved as `mouth_crop.mp4`.
3. Load the checkpoint into the backbone, encode, then beam search (40 candidates, ctc_weight 0.1).
4. Print the **Result line**.

The first run takes longest: it loads ~4 GB of weights, and MediaPipe does about 20–50 ms per frame on the CPU.

In [ ]:
import re, sys, time, subprocess

def transcribe(video, modality, extra=()):
    cmd = [sys.executable, "demo.py", f"video={video}", f"model.pretrained_model_path={CKPT}",
           f"model/backbone={BACKBONE}", f"modality={modality}", *extra]
    t0 = time.time()
    r = subprocess.run(cmd, cwd="/content/usr2", capture_output=True, text=True)
    m = re.search(r"Result\s*:\s*(.*)", r.stdout)
    if not m:
        print(r.stdout[-2000:]); print(r.stderr[-4000:])
        raise RuntimeError("No Result line. Read the LAST line of the error above, then see Troubleshooting.")
    return m.group(1).strip(), time.time() - t0

results = {}
text, secs = transcribe(CLIP, "av")
results["av"] = text
print(f"[av] {secs:.0f} s  ->  {text}")

### 6b. Look at the plate: the mouth crop
**Chapter 7's rule:** if the words come out wrong, watch this file first. **If the mouth isn't steady and centred, the chef never had a chance.** (The file uses a codec browsers can't play, so we re-encode and enlarge it.)

In [ ]:
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/usr2/mouth_crop.mp4",
                "-vf", "scale=264:264:flags=neighbor", "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "/content/work/mouth_view.mp4"], check=True)
show_video("/content/work/mouth_view.mp4", width=264)

# Step 7: Same clip, one sense at a time, then the blood test · 3–8 min

**Word error rate** = (substituted + deleted + inserted words) ÷ number of words in the label. Lower is better, and it can exceed 100 % if the model invents words. The alignment printout shows the **visemes** in action: lips-only mistakes are often words that *look* the same (the paper's own example: *cautious → conscious*).

In [ ]:
for mod in ["a", "v"]:
    text, secs = transcribe(CLIP, mod)
    results[mod] = text
    print(f"[{mod}] {secs:.0f} s  ->  {text}")

In [ ]:
import re, jiwer

def norm(s):
    s = s.upper().replace("’", "'")
    s = re.sub(r"[^A-Z0-9' ]+", " ", s)
    return " ".join(s.split())

ref = norm(REFERENCE)
print("LABEL:", ref, f"({len(ref.split())} words)\n")
names = {"a": "audio only", "v": "lips only", "av": "audio + lips"}
for mod in ["a", "v", "av"]:
    out = jiwer.process_words(ref, norm(results[mod]))
    print(f"=== {mod:>2} ({names[mod]}): WER {out.wer*100:.1f} %  "
          f"(sub {out.substitutions}, del {out.deletions}, ins {out.insertions})")
    print(jiwer.visualize_alignment(out, show_measures=False))

# Step 8: The train goes over (babble noise) · 5–10 min

**This is what's at stake for the cafe.** We mix *babble* (many people talking at once, the paper's test noise from NOISEX) into the clip's audio at a chosen **signal-to-noise ratio**. **0 dB** means the crowd is as loud as the speaker. Then we compare `a` with `av`. The paper (Base model, LRS3, 0 dB): audio only **44.0 %** vs audio + lips **14.0 %**.

The cell downloads the authors' babble file. If that fails, it builds a stand-in babble from six time-shifted copies of the clip's own audio (a crowd of clones). Same idea, less realistic.

In [ ]:
import numpy as np, scipy.io.wavfile as wavfile, gdown, os
SNR_DB = 0          # try 5, 0, -5

subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", CLIP, "-vn", "-ac", "1", "-ar", "16000",
                "/content/work/speech.wav"], check=True)
sr, speech = wavfile.read("/content/work/speech.wav")
speech = speech.astype(np.float32)

noise = None
try:
    if not os.path.exists("/content/work/babble.npy"):
        gdown.download(id="1d8D6FcsftCotnE14nu9C2q1v6N0KnYCC", output="/content/work/babble.npy", quiet=True)
    noise = np.load("/content/work/babble.npy", allow_pickle=True).astype(np.float32).ravel()
    print("Using the authors' babble noise.")
except Exception as e:
    print("Babble download failed (", e, "); using a crowd of time-shifted clones instead.")
    rng = np.random.default_rng(0)
    noise = sum(np.roll(speech, int(rng.integers(len(speech)))) for _ in range(6))

rng = np.random.default_rng(42)
if len(noise) < len(speech):
    noise = np.tile(noise, len(speech) // len(noise) + 1)
off = int(rng.integers(0, len(noise) - len(speech) + 1))
noise = noise[off:off + len(speech)]
p_s, p_n = np.mean(speech**2), np.mean(noise**2) + 1e-9
noise *= np.sqrt(p_s / (p_n * 10 ** (SNR_DB / 10)))
mix = speech + noise
mix *= min(1.0, 32000 / (np.abs(mix).max() + 1e-9))     # avoid clipping; SNR unchanged
wavfile.write("/content/work/noisy.wav", sr, mix.astype(np.int16))

NOISY = f"/content/work/clip_noisy_{SNR_DB}dB.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", CLIP, "-i", "/content/work/noisy.wav",
                "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "aac", "-shortest", NOISY], check=True)
print(f"Noisy clip at {SNR_DB} dB SNR. Listen: can YOU still follow it?")
show_video(NOISY)

In [ ]:
for mod in ["a", "av"]:
    text, secs = transcribe(NOISY, mod)
    w = jiwer.wer(ref, norm(text)) * 100
    clean = jiwer.wer(ref, norm(results[mod])) * 100
    print(f"[{mod:>2}] clean WER {clean:5.1f} %  ->  {SNR_DB} dB WER {w:5.1f} %   | {text}")

**Read your result like Sami would:**
* If `av` stays close to its clean score while `a` jumps, you've reproduced the paper's main promise: **the lips win when the train goes over.**
* If both stay good, the noise wasn't hard enough for this speaker. Set `SNR_DB = -5` and rerun the two cells above.
* If `av` collapses too, check the mouth crop (Step 6b) and the clip choice: the visual half may be **out of distribution** (side view, low resolution, poor light).

### Optional: the latency knob
The beam keeps 40 candidates. The paper says USR 2.0 stays strong with small beams. Try 10 (or 1, which is greedy) and compare the time and the text:

In [ ]:
text, secs = transcribe(CLIP, "av", extra=["decode.beam_size=10"])
print(f"[av, beam 10] {secs:.0f} s -> {text}")

# Step 9: Close the kitchen · 1 min

**Chapter 5's rule:** a face video is **personal data**, so delete clips once the words are out. Then **Runtime → Disconnect and delete runtime** to give the GPU back. Everything on this machine disappears anyway when it's recycled.

In [ ]:
!rm -rf /content/work /content/usr2/mouth_crop.mp4
print("Clips deleted. Now: Runtime > Disconnect and delete runtime.")

---
# Troubleshooting: read the last line of the traceback

The four complaints from Chapter 3, plus Colab's own:

| Last line says… | Meaning (Noor) | Fix |
|---|---|---|
| `FileNotFoundError` | wrong path | Rerun Steps 4 and 5c; check `/content/ckpt` and `/content/work` in the Files panel. |
| `ModuleNotFoundError` / `AttributeError ... read_video` | shelf missing an ingredient | Rerun Step 3, restart the session, then run Step 3b. |
| `size mismatch` / `Missing key(s) in state_dict` | skeleton doesn't match the brain | `MODEL` and `BACKBONE` must match (Step 4). |
| `Could not detect a face in enough frames` | no face found | Pick another segment: frontal, well lit, mouth visible. |
| `CUDA out of memory` / "Your session crashed after using all available RAM" | kitchen too small | Shorten the clip, or use `MODEL = "baseplus"`. |
| `Too many users have viewed or downloaded this file` | Drive limit | Manual download (see Step 4) or `"baseplus"`. |
| `Sign in to confirm you're not a bot` | YouTube blocks Colab | Step 5b, Plan B (upload). |

Note: the docstring at the top of `demo.py` shows `--video ... --checkpoint ...` flags. Those don't work. The script uses Hydra-style `name=value` arguments, as in the README and in the `transcribe()` helper above.